<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/Embrapa_ResNet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()  # Upload kaggle.json

!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset
!kaggle datasets download -d poornimasinghthakur/embrapa
!unzip -q embrapa.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/poornimasinghthakur/embrapa
License(s): unknown
100% 1.05G/1.05G [00:05<00:00, 235MB/s]
100% 1.05G/1.05G [00:05<00:00, 194MB/s]


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing import image_dataset_from_directory

img_size = (224, 224)
batch_size = 32


tmp_train_ds = image_dataset_from_directory(
    "/content/XDB/train",
    image_size=img_size,
    batch_size=batch_size
)
class_names = tmp_train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

# Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

# Augment training data
train_ds = tmp_train_ds.map(lambda x, y: (data_augmentation(x), y))

# Validation & Test datasets
val_ds = image_dataset_from_directory(
    "/content/XDB/val",
    image_size=img_size,
    batch_size=batch_size
)
test_ds = image_dataset_from_directory(
    "/content/XDB/test",
    image_size=img_size,
    batch_size=batch_size
)

# Convert labels to one-hot encoding for CategoricalCrossentropy
def one_hot_map(images, labels):
    return images, tf.one_hot(labels, depth=num_classes)

train_ds_onehot = train_ds.map(one_hot_map)
val_ds_onehot = val_ds.map(one_hot_map)
test_ds_onehot = test_ds.map(one_hot_map)

# Prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds_onehot = train_ds_onehot.prefetch(AUTOTUNE)
val_ds_onehot = val_ds_onehot.prefetch(AUTOTUNE)
test_ds_onehot = test_ds_onehot.prefetch(AUTOTUNE)

Found 29607 files belonging to 93 classes.
Classes: ['AlgodтФЬ├║o (Cotton) - Mancha de Mirotecio (Myrothecium leaf spot) - Cropped', 'AlgodтФЬ├║o (Cotton) - Mela (Soreshin) - Cropped', 'AlgodтФЬ├║o (Cotton) - Ramularia (Areolate Mildew) - Cropped', 'Arroz (Rice) - Brusone (Rice Blast) - Cropped', 'Arroz (Rice) - Escaldadura (Leaf Scald) - Cropped', 'CafтФЬтМР (Coffee) - BichoMineiro (Leaf Miner) - Cropped', 'CafтФЬтМР (Coffee) - Cercosporiose (Cercospora Leaf Spot) - Cropped', 'CafтФЬтМР (Coffee) - Ferrugem (Rust) - Cropped', 'CafтФЬтМР (Coffee) - Mancha Aureolada (Bacterial Blight) - Cropped', 'CafтФЬтМР (Coffee) - Mancha Mantegosa (Blister Spot) - Cropped', 'CafтФЬтМР (Coffee) - Phoma (Brown leaf spot) - Cropped', 'Cajueiro (Cashew Tree) - Alga (Algae) - Cropped', 'Cajueiro (Cashew Tree) - Antracnose (Anthracnose) - Cropped', 'Cajueiro (Cashew Tree) - Mancha Angular (Angular Leaf Spot) - Cropped', 'Cajueiro (Cashew Tree) - Mofo Preto (Black Mould) - Cropped', 'Cajueiro (Cashew Tree) 

In [ ]:
base_model = ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation='softmax')
])

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [ ]:
history = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=10
)

Epoch 1/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 474s 495ms/step - accuracy: 0.3667 - loss: 3.3890 - val_accuracy: 0.6987 - val_loss: 2.1102
Epoch 2/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 451s 449ms/step - accuracy: 0.5804 - loss: 2.4525 - val_accuracy: 0.7395 - val_loss: 1.9650
Epoch 3/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 414s 447ms/step - accuracy: 0.6091 - loss: 2.3473 - val_accuracy: 0.7391 - val_loss: 1.9655
Epoch 4/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 415s 448ms/step - accuracy: 0.6201 - loss: 2.3144 - val_accuracy: 0.7518 - val_loss: 1.9208
Epoch 5/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 417s 450ms/step - accuracy: 0.6243 - loss: 2.3049 - val_accuracy: 0.7600 - val_loss: 1.9233
Epoch 6/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 415s 448ms/step - accuracy: 0.6322 - loss: 2.2822 - val_accuracy: 0.7659 - val_loss: 1.8937
Epoch 7/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 421s 454ms/step - accuracy: 0.6305 - loss: 2.2737 - val_accuracy: 0.7524 - val_loss: 1.9323
Epoch 8/10
926/926 ━━━━━━━━━━━━━━━━━━━━ 413s 446ms/step - accuracy: 0.6377 -

In [ ]:
base_model.trainable = True
fine_tune_at = 50
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=20,
    initial_epoch=10
)

Epoch 11/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 555s 550ms/step - accuracy: 0.3828 - loss: 3.2583 - val_accuracy: 0.7784 - val_loss: 1.8727
Epoch 12/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 491s 530ms/step - accuracy: 0.6748 - loss: 2.1703 - val_accuracy: 0.8245 - val_loss: 1.7353
Epoch 13/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 487s 526ms/step - accuracy: 0.7311 - loss: 2.0188 - val_accuracy: 0.8486 - val_loss: 1.6680
Epoch 14/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 489s 528ms/step - accuracy: 0.7632 - loss: 1.9232 - val_accuracy: 0.8618 - val_loss: 1.6190
Epoch 15/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 499s 525ms/step - accuracy: 0.7862 - loss: 1.8599 - val_accuracy: 0.8700 - val_loss: 1.5824
Epoch 16/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 502s 526ms/step - accuracy: 0.8061 - loss: 1.8110 - val_accuracy: 0.8805 - val_loss: 1.5528
Epoch 17/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 513s 537ms/step - accuracy: 0.8194 - loss: 1.7623 - val_accuracy: 0.8846 - val_loss: 1.5289
Epoch 18/20
926/926 ━━━━━━━━━━━━━━━━━━━━ 488s 527ms/step - accuracy: 

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize

TEMPERATURE = 2.0

def predict_with_temperature(model, dataset, T=1.0):
    all_probs, all_labels = [], []
    for images, labels in dataset:
        logits = model(images, training=False)
        scaled_logits = logits / T
        probs = tf.nn.softmax(scaled_logits).numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_probs)

y_true, y_probs = predict_with_temperature(model, test_ds_onehot, T=TEMPERATURE)
y_pred = np.argmax(y_probs, axis=1)

y_true_bin = label_binarize(y_true.argmax(axis=1), classes=range(num_classes))

accuracy = accuracy_score(y_true.argmax(axis=1), y_pred)
precision = precision_score(y_true.argmax(axis=1), y_pred, average='weighted')
recall = recall_score(y_true.argmax(axis=1), y_pred, average='weighted')
f1 = f1_score(y_true.argmax(axis=1), y_pred, average='weighted')
kappa = cohen_kappa_score(y_true.argmax(axis=1), y_pred)
auc = roc_auc_score(y_true_bin, y_probs, average='macro', multi_class='ovr')

print("\n Evaluation Metrics After Calibration:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Kappa    : {kappa:.4f}")

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



 Evaluation Metrics After Calibration:
Accuracy : 0.8976
Precision: 0.8929
Recall   : 0.8976
F1 Score : 0.8904
AUC      : 0.9980
Kappa    : 0.8936
